# European Bank Customer Churn Analysis
## Customer Segmentation, Churn Patterns and Predictive Modelling

**Dataset:** European Bank customer records  
**Target variable:** `Exited` (1 = churned, 0 = retained)

### Notebook workflow
1. Load the CSV dataset
2. Inspect and clean the data
3. Explore customer churn and retention
4. Compare churn across customer segments
5. Visualise key patterns and correlations
6. Train Logistic Regression and Random Forest models
7. Evaluate model performance and feature importance
8. Export analysis tables and predictions

**How to use:** Run the cells from top to bottom. In Google Colab, the upload cell will ask you to select your CSV file. In local Jupyter, put the CSV in the notebook folder or update `CSV_PATH`.


## 1. Install and import required libraries

In [ ]:
# Install only if required (safe to run in Google Colab/Jupyter)
%pip -q install pandas numpy matplotlib seaborn scikit-learn openpyxl

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42


## 2. Upload and load the dataset

In [ ]:
# Google Colab: choose the CSV file when prompted.
# Local Jupyter: set CSV_PATH to your CSV file's location.
try:
    from google.colab import files
    uploaded = files.upload()
    csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
    if not csv_files:
        raise FileNotFoundError("No CSV file was uploaded. Please upload the European Bank CSV.")
    CSV_PATH = csv_files[0]
except ImportError:
    CSV_PATH = "European_Bank (1)(1).csv"  # Change this if your CSV has a different name.

df = pd.read_csv(CSV_PATH)
df.columns = df.columns.astype(str).str.strip()

print("Dataset loaded successfully")
print("File:", CSV_PATH)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
display(df.head())


## 3. Data understanding and quality checks

In [ ]:
print("Dataset shape:", df.shape)
print("\\nColumn names:")
print(df.columns.tolist())

data_dictionary = pd.DataFrame({
    "Column": df.columns,
    "Data Type": [str(df[c].dtype) for c in df.columns],
    "Missing Values": [int(df[c].isna().sum()) for c in df.columns],
    "Unique Values": [int(df[c].nunique(dropna=True)) for c in df.columns]
})
display(data_dictionary)

print("Exact duplicate rows:", int(df.duplicated().sum()))


## 4. Data cleaning and target validation

In [ ]:
# Remove exact duplicate records and standardise the target.
df = df.drop_duplicates().copy()

required_columns = [
    "Exited", "Geography", "Age", "Gender", "CreditScore",
    "Tenure", "Balance", "NumOfProducts", "HasCrCard",
    "IsActiveMember", "EstimatedSalary"
]
missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(
        "The dataset is missing expected columns: " + ", ".join(missing_columns)
    )

# Convert target to numeric 0/1.
if not pd.api.types.is_numeric_dtype(df["Exited"]):
    target_text = df["Exited"].astype(str).str.strip().str.lower()
    target_map = {
        "yes": 1, "true": 1, "churned": 1, "exited": 1, "1": 1,
        "no": 0, "false": 0, "retained": 0, "active": 0, "0": 0
    }
    df["Exited"] = target_text.map(target_map)

df["Exited"] = pd.to_numeric(df["Exited"], errors="coerce")
df = df.dropna(subset=["Exited"]).copy()
df = df[df["Exited"].isin([0, 1])].copy()
df["Exited"] = df["Exited"].astype(int)

# Convert known numeric fields.
numeric_columns = [
    "Year", "CreditScore", "Age", "Tenure", "Balance",
    "NumOfProducts", "HasCrCard", "IsActiveMember",
    "EstimatedSalary"
]
for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Rows after cleaning:", len(df))
print("Missing values after cleaning:")
display(df.isna().sum().to_frame("Missing Values"))
print("Target values:", sorted(df["Exited"].unique().tolist()))


## 5. Overall customer churn and retention

In [ ]:
total_customers = len(df)
churned_customers = int((df["Exited"] == 1).sum())
retained_customers = int((df["Exited"] == 0).sum())
churn_rate = churned_customers / total_customers * 100
retention_rate = retained_customers / total_customers * 100

overall_summary = pd.DataFrame({
    "Metric": [
        "Total Customers", "Churned Customers", "Retained Customers",
        "Churn Rate (%)", "Retention Rate (%)"
    ],
    "Value": [
        total_customers, churned_customers, retained_customers,
        round(churn_rate, 2), round(retention_rate, 2)
    ]
})
display(overall_summary)

plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="Exited")
plt.title("Customer Churn Distribution")
plt.xlabel("Customer Status (0 = Retained, 1 = Churned)")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()


## 6. Churn analysis by customer segments

In [ ]:
def churn_segment_summary(data, column):
    summary = (
        data.groupby(column, dropna=False)["Exited"]
        .agg(Customers="size", Churned="sum", Churn_Rate="mean")
        .reset_index()
    )
    summary["Retained"] = summary["Customers"] - summary["Churned"]
    summary["Churn Rate (%)"] = (summary["Churn_Rate"] * 100).round(2)
    return summary.drop(columns=["Churn_Rate"])

segment_columns = [
    "Geography", "Gender", "Tenure", "NumOfProducts",
    "HasCrCard", "IsActiveMember"
]
segment_results = {}

for col in segment_columns:
    if col in df.columns:
        print(f"Churn analysis by {col}")
        result = churn_segment_summary(df, col)
        segment_results[col] = result
        display(result)


## 7. Age-group analysis

In [ ]:
age_df = df.dropna(subset=["Age"]).copy()
age_df["Age Group"] = pd.cut(
    age_df["Age"],
    bins=[0, 25, 35, 45, 55, 65, np.inf],
    labels=["25 and below", "26–35", "36–45", "46–55", "56–65", "Above 65"],
    include_lowest=True
)
age_summary = churn_segment_summary(age_df, "Age Group")
display(age_summary)

plt.figure(figsize=(9, 5))
sns.barplot(data=age_summary, x="Age Group", y="Churn Rate (%)")
plt.title("Churn Rate by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 8. Credit score and balance segmentation

In [ ]:
credit_df = df.dropna(subset=["CreditScore"]).copy()
credit_df["Credit Score Group"] = pd.cut(
    credit_df["CreditScore"],
    bins=[-np.inf, 579, 669, 739, 799, np.inf],
    labels=["Poor (≤579)", "Fair (580–669)", "Good (670–739)",
            "Very Good (740–799)", "Excellent (800+)"]
)
credit_summary = churn_segment_summary(credit_df, "Credit Score Group")
display(credit_summary)

balance_df = df.dropna(subset=["Balance"]).copy()
balance_df["Balance Group"] = pd.cut(
    balance_df["Balance"],
    bins=[-0.01, 0, 50000, 100000, 150000, np.inf],
    labels=["Zero", "1–50k", "50k–100k", "100k–150k", "Above 150k"]
)
balance_summary = churn_segment_summary(balance_df, "Balance Group")
display(balance_summary)


## 9. Visual exploration of selected variables

In [ ]:
plot_columns = [
    ("Geography", "Churn Rate by Geography"),
    ("Gender", "Churn Rate by Gender"),
    ("NumOfProducts", "Churn Rate by Number of Products"),
    ("IsActiveMember", "Churn Rate by Active Membership"),
    ("HasCrCard", "Churn Rate by Credit Card Ownership")
]

for col, title in plot_columns:
    if col in df.columns:
        plot_data = churn_segment_summary(df, col)
        plt.figure(figsize=(7, 4))
        sns.barplot(data=plot_data, x=col, y="Churn Rate (%)")
        plt.title(title)
        plt.xlabel(col)
        plt.ylabel("Churn Rate (%)")
        plt.tight_layout()
        plt.show()


## 10. Numerical comparison: retained vs churned customers

In [ ]:
comparison_columns = [
    c for c in [
        "CreditScore", "Age", "Tenure", "Balance",
        "NumOfProducts", "EstimatedSalary"
    ] if c in df.columns
]
retained_vs_churned = df.groupby("Exited")[comparison_columns].mean().T
retained_vs_churned.columns = ["Retained (0)", "Churned (1)"]
display(retained_vs_churned)


## 11. Correlation analysis

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
plt.figure(figsize=(11, 8))
sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix of Numerical Variables")
plt.tight_layout()
plt.show()

if "Exited" in numeric_df.columns:
    churn_corr = numeric_df.corr()["Exited"].drop("Exited").sort_values(ascending=False)
    print("Correlation with Exited:")
    display(churn_corr.to_frame("Correlation with Exited"))


## 12. Prepare data for predictive modelling

In [ ]:
# Exclude identifiers and names to avoid learning customer identity.
# Exclude Year because it is a record/reference field in this dataset.
identifier_columns = [
    c for c in ["RowNumber", "CustomerId", "CustomerID", "Surname", "Year"]
    if c in df.columns
]

X = df.drop(columns=["Exited"] + identifier_columns, errors="ignore")
y = df["Exited"].astype(int)

# Remove columns that contain no usable information.
X = X.dropna(axis=1, how="all")
X = X.replace([np.inf, -np.inf], np.nan)

numeric_features = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

print("Predictor columns:", X.columns.tolist())
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Target distribution:")
display(y.value_counts().rename(index={0: "Retained", 1: "Churned"}).to_frame("Customers"))


## 13. Train Logistic Regression and Random Forest

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

trained_models = {}
predictions = {}
probabilities = {}
metric_rows = []

for model_name, estimator in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    trained_models[model_name] = pipeline
    predictions[model_name] = y_pred
    probabilities[model_name] = y_prob

    metric_rows.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

model_comparison = pd.DataFrame(metric_rows).set_index("Model")
display(model_comparison.round(4))


## 14. Confusion matrices and classification reports

In [ ]:
for model_name in trained_models:
    print("\\n", model_name)
    print(classification_report(
        y_test, predictions[model_name],
        target_names=["Retained", "Churned"],
        zero_division=0
    ))

    cm = confusion_matrix(y_test, predictions[model_name])
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Retained", "Churned"]
    )
    disp.plot(cmap="Blues", values_format="d")
    plt.title(f"Confusion Matrix - {model_name}")
    plt.tight_layout()
    plt.show()


## 15. ROC curve comparison

In [ ]:
plt.figure(figsize=(7, 6))
for model_name, y_prob in probabilities.items():
    RocCurveDisplay.from_predictions(
        y_test, y_prob, name=model_name, ax=plt.gca()
    )
plt.plot([0, 1], [0, 1], linestyle="--", label="Random baseline")
plt.title("ROC Curve Comparison")
plt.tight_layout()
plt.show()


## 16. Random Forest feature importance

In [ ]:
rf_pipeline = trained_models["Random Forest"]
rf_model = rf_pipeline.named_steps["model"]
feature_names = rf_pipeline.named_steps["preprocessor"].get_feature_names_out()

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

display(feature_importance.head(15))

plt.figure(figsize=(9, 6))
sns.barplot(
    data=feature_importance.head(15),
    x="Importance",
    y="Feature"
)
plt.title("Top 15 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


## 17. Generate customer churn predictions

In [ ]:
# Use the Random Forest model to estimate churn probability for the full dataset.
full_probabilities = rf_pipeline.predict_proba(X)[:, 1]
full_predictions = (full_probabilities >= 0.50).astype(int)

prediction_output = df.copy()
prediction_output["Predicted_Churn_Probability"] = full_probabilities
prediction_output["Predicted_Exited"] = full_predictions
prediction_output["Predicted_Status"] = np.where(
    full_predictions == 1, "Likely to Churn", "Likely to Stay"
)

display(prediction_output[
    ["Predicted_Churn_Probability", "Predicted_Exited", "Predicted_Status"]
].head())

print("Predicted churn-risk categories:")
display(prediction_output["Predicted_Status"].value_counts().to_frame("Customers"))


## 18. Export project results

In [ ]:
output_dir = Path("bank_churn_outputs")
output_dir.mkdir(exist_ok=True)

overall_summary.to_csv(output_dir / "overall_churn_summary.csv", index=False)
data_dictionary.to_csv(output_dir / "data_quality_report.csv", index=False)
model_comparison.to_csv(output_dir / "model_comparison.csv")

for segment_name, result in segment_results.items():
    result.to_csv(output_dir / f"churn_by_{segment_name.lower()}.csv", index=False)

age_summary.to_csv(output_dir / "churn_by_age_group.csv", index=False)
credit_summary.to_csv(output_dir / "churn_by_credit_score_group.csv", index=False)
balance_summary.to_csv(output_dir / "churn_by_balance_group.csv", index=False)
feature_importance.to_csv(output_dir / "random_forest_feature_importance.csv", index=False)
prediction_output.to_csv(output_dir / "customer_churn_predictions.csv", index=False)

excel_path = output_dir / "European_Bank_Churn_Analysis_Results.xlsx"
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    overall_summary.to_excel(writer, sheet_name="Overall Summary", index=False)
    data_dictionary.to_excel(writer, sheet_name="Data Quality", index=False)
    model_comparison.to_excel(writer, sheet_name="Model Comparison")
    age_summary.to_excel(writer, sheet_name="Age Groups", index=False)
    credit_summary.to_excel(writer, sheet_name="Credit Score", index=False)
    balance_summary.to_excel(writer, sheet_name="Balance Groups", index=False)
    feature_importance.to_excel(writer, sheet_name="Feature Importance", index=False)
    prediction_output.to_excel(writer, sheet_name="Predictions", index=False)

print("Results saved in:", output_dir.resolve())
print("Excel workbook:", excel_path.resolve())


## 19. Key findings – report writing guide

After running the notebook, use the actual output tables and charts to write your findings. Discuss:

- Overall customer churn and retention rates.
- Customer groups with comparatively higher churn rates.
- Differences between retained and churned customers.
- The most important predictors identified by the Random Forest model.
- Comparison of Logistic Regression and Random Forest using accuracy, precision, recall, F1-score and ROC-AUC.
- Practical retention actions supported by the analysis.

**Important:** Feature importance and correlation show association/predictive usefulness, not proof of causation. Model performance is based on the held-out test sample and may change with a different dataset or split.
